# Crack graph — labeled

Build the graph from hand-labelled contours and sweep the edge-type energy weights to find minimum-energy crack paths.

In [ ]:
from combra import graph, data

import copy
import numpy as np

# Labeled data input

In [ ]:
labeled_cnts, labels = data.crack_labeled_contours()

# image is used only for shape
_, image = data.crack_images()[0]

border = 30
eps = 300

(entry_nodes,
 exit_nodes,
 img_contours,
 img_preprocessed_final,
 cnts,
 nodes_metadata) = graph.preprocess_graph_image(
    image, border=border, disk=5, labeled_cnts=labeled_cnts, labels=labels)

g, _ = graph.create_crack_graph(
    img_preprocessed_final.shape, cnts, nodes_metadata, eps=eps, labels=labels)

# Graph plot

In [ ]:
save = False
# save = True

g_cleaned = copy.deepcopy(g)

graph.graph_plot(g_cleaned, img_contours[:500],
                 name='wc_cv/cv/wc_co_labeled_graph.html', save=save, node_size=30)

# Weight-grid sweep

edge types: `0` Co, `1` WC-Co, `2` WC, `3` WC-WC

In [ ]:
%%time

# WC+8Co_5_crack hand labeled, no WC movement
entry_nodes = [636, 635, 625, 639]
exit_nodes = [399, 398, 1092]

first_k_paths = 10
parallel = True
workers = 23

# drop WC edges, then sweep WC-Co (rows) x WC-WC (cols) with Co fixed at 10
g_cleaned = graph.remove_edges_of_type(copy.deepcopy(g), 2)
energy_conf = graph.build_energy_grid(20, 20, const={0: 10}, row_key=1, col_key=3)

energies = graph.get_energies(
    energy_conf, g_cleaned, cnts, nodes_metadata,
    entry_nodes, exit_nodes,
    first_k_paths=first_k_paths, parallel=parallel, workers=workers)

In [ ]:
name = 'WC+8Co_5_crack_optimized_e_labeled_wc_co_20_wc_wc_20_no_wc_co_10.jpg'
save = False
path_index = 0

graph.plot_optimized_energies(
    energies, path_index=path_index, N=5, M=5,
    y_label='wc_co_e', x_label='wc_wc_e', name=name, save=save)

# Fixed paths energies

In [ ]:
%%time

entry_nodes = [636, 635, 625, 639]
exit_nodes = [399, 398, 1092]

workers = 23
np.random.seed(51)

# base paths energies (single config cell)
energy_conf = graph.build_energy_grid(1, 1, const={0: 15, 1: 15, 2: 20, 3: 0})
energies_paths = graph.get_energies(
    energy_conf, g, cnts, nodes_metadata, entry_nodes, exit_nodes,
    first_k_paths=1, parallel=True, workers=workers)

# recalculate along the fixed paths: Co (rows) x WC-Co (cols), WC=20, WC-WC=0
energy_conf = graph.build_energy_grid(20, 20, const={2: 20, 3: 0}, row_key=0, col_key=1)
energies_paths_recalculated = graph.get_energies(
    energy_conf, g, cnts, nodes_metadata, entry_nodes, exit_nodes,
    first_k_paths=1, parallel=False, workers=workers,
    recalculate_paths=energies_paths)

In [ ]:
name = 'wc_cv/cv/WC+8Co_5_crack_fixed_optimized_e_labeled_no_wc_10.jpg'
save = False
path_index = 0

graph.plot_optimized_energies(
    energies_paths_recalculated, path_index=path_index, N=5, M=5,
    y_label='co_co_e', x_label='wc_co_e', name=name, save=save, fixed_paths=True)

# Optimized path overlay

In [ ]:
# Draw the energy-optimised paths for one grid cell of the `energies` sweep.
graph.plot_optimized_paths(g_cleaned, energies, img_contours, param_1=10, param_2=10)